In [25]:
import pandas as pd
import numpy as np
import os

In [189]:
TEST = 'googl_ping_test_1.csv'
TEST_OUTPUT = 'output/ping/googl_ping_test_1_processed.csv'

In [190]:
df = pd.read_csv(TEST)

In [191]:
# Get time span before filtering
time_span_raw = df['Time'].max() - df['Time'].min()
print(f"Time span (before processing): {time_span_raw} seconds")

Time span (before processing): 301.0265253 seconds


In [192]:
df.head()

,No.,Time,Source,Destination,Protocol,Length,Info
0,142,15.549100,192.168.0.200,216.239.38.120,ICMP,74,"Echo (ping) request id=0x0001, seq=412/39937,..."
1,143,15.595491,216.239.38.120,192.168.0.200,ICMP,74,"Echo (ping) reply id=0x0001, seq=412/39937,..."
2,150,16.552307,192.168.0.200,216.239.38.120,ICMP,74,"Echo (ping) request id=0x0001, seq=413/40193,..."
3,151,16.622112,216.239.38.120,192.168.0.200,ICMP,74,"Echo (ping) reply id=0x0001, seq=413/40193,..."
4,159,17.565556,192.168.0.200,216.239.38.120,ICMP,74,"Echo (ping) request id=0x0001, seq=414/40449,..."


In [193]:
df['Protocol'].value_counts()

Protocol
ICMP    379
Name: count, dtype: int64

In [194]:
# Use this if you want to filter by protocol and port (e.g., HTTP/HTTPS)

# mask_protocol = df['Protocol'].isin(['TCP', 'UDP'])  # Adjust protocol names as needed [e.g., 'TCP', 'UDP']
# mask_port = df['Info'].str.contains(r'\b80\b|\b443\b', na=False)
# df_filtered = df[mask_protocol | mask_port]

df_filtered = df

In [195]:
df_filtered

,No.,Time,Source,Destination,Protocol,Length,Info
0,142,15.549100,192.168.0.200,216.239.38.120,ICMP,74,"Echo (ping) request id=0x0001, seq=412/39937,..."
1,143,15.595491,216.239.38.120,192.168.0.200,ICMP,74,"Echo (ping) reply id=0x0001, seq=412/39937,..."
2,150,16.552307,192.168.0.200,216.239.38.120,ICMP,74,"Echo (ping) request id=0x0001, seq=413/40193,..."
3,151,16.622112,216.239.38.120,192.168.0.200,ICMP,74,"Echo (ping) reply id=0x0001, seq=413/40193,..."
4,159,17.565556,192.168.0.200,216.239.38.120,ICMP,74,"Echo (ping) request id=0x0001, seq=414/40449,..."
...,...,...,...,...,...,...,...
374,644,314.358496,216.239.38.120,192.168.0.200,ICMP,74,"Echo (ping) reply id=0x0001, seq=611/25346,..."
375,645,315.310886,192.168.0.200,216.239.38.120,ICMP,74,"Echo (ping) request id=0x0001, seq=612/25602,..."
376,646,315.377488,216.239.38.120,192.168.0.200,ICMP,74,"Echo (ping) reply id=0x0001, seq=612/25602,..."
377,647,316.316049,192.168.0.200,216.239.38.120,ICMP,74,"Echo (ping) request id=0x0001, seq=613/25858,..."


In [196]:
df_filtered.value_counts('Protocol')

Protocol
ICMP    379
Name: count, dtype: int64

In [197]:
df_filtered.head()

,No.,Time,Source,Destination,Protocol,Length,Info
0,142,15.549100,192.168.0.200,216.239.38.120,ICMP,74,"Echo (ping) request id=0x0001, seq=412/39937,..."
1,143,15.595491,216.239.38.120,192.168.0.200,ICMP,74,"Echo (ping) reply id=0x0001, seq=412/39937,..."
2,150,16.552307,192.168.0.200,216.239.38.120,ICMP,74,"Echo (ping) request id=0x0001, seq=413/40193,..."
3,151,16.622112,216.239.38.120,192.168.0.200,ICMP,74,"Echo (ping) reply id=0x0001, seq=413/40193,..."
4,159,17.565556,192.168.0.200,216.239.38.120,ICMP,74,"Echo (ping) request id=0x0001, seq=414/40449,..."


In [ ]:
df_filtered['Time1'] = df['Time']
df_filtered['Time2'] = df_filtered['Time1'].shift(-1)

df_filtered['Delay'] = df_filtered['Time2'] - df_filtered['Time1']
df_filtered['Delay1'] = abs(df_filtered['Delay'] - df_filtered['Delay'].shift(-1))

df_filtered['Delay2'] = abs(df_filtered['Delay1'].shift(-1))
df_filtered['Jitter'] = np.abs(df_filtered['Delay2'] - df_filtered['Delay1'])

df_filtered.head()

,No.,Time,Source,Destination,Protocol,Length,Info,Time1,Time2,Delay,Delay1,Delay2,Jitter
0,142,15.549100,192.168.0.200,216.239.38.120,ICMP,74,"Echo (ping) request id=0x0001, seq=412/39937,...",15.549100,15.595491,0.046391,-0.910426,0.887012,1.797438
1,143,15.595491,216.239.38.120,192.168.0.200,ICMP,74,"Echo (ping) reply id=0x0001, seq=412/39937,...",15.595491,16.552307,0.956817,0.887012,-0.873639,1.760651
2,150,16.552307,192.168.0.200,216.239.38.120,ICMP,74,"Echo (ping) request id=0x0001, seq=413/40193,...",16.552307,16.622112,0.069805,-0.873639,0.864440,1.738079
3,151,16.622112,216.239.38.120,192.168.0.200,ICMP,74,"Echo (ping) reply id=0x0001, seq=413/40193,...",16.622112,17.565556,0.943444,0.864440,-0.849893,1.714333
4,159,17.565556,192.168.0.200,216.239.38.120,ICMP,74,"Echo (ping) request id=0x0001, seq=414/40449,...",17.565556,17.644559,0.079004,-0.849893,0.883103,1.732996


In [199]:
request_mask = df_filtered['Info'].str.contains(r'^Echo \(ping\) request', case=False)
reply_mask = df_filtered['Info'].str.contains(r'^Echo \(ping\) reply', case=False)

request_df_filtered = df_filtered[request_mask]
reply_df_filtered = df_filtered[reply_mask]

In [200]:
# Reorder columns - No first, then Time1, Time2, etc
cols_order = ['No.', 'Time', 'Time1', 'Time2', 'Delay', 'Delay1', 'Delay2', 'Jitter']
other_cols = [col for col in df_filtered.columns if col not in cols_order]
df_filtered = df_filtered[cols_order + other_cols]
df_filtered.head()

,No.,Time,Time1,Time2,Delay,Delay1,Delay2,Jitter,Source,Destination,Protocol,Length,Info
0,142,15.549100,15.549100,15.595491,0.046391,-0.910426,0.887012,1.797438,192.168.0.200,216.239.38.120,ICMP,74,"Echo (ping) request id=0x0001, seq=412/39937,..."
1,143,15.595491,15.595491,16.552307,0.956817,0.887012,-0.873639,1.760651,216.239.38.120,192.168.0.200,ICMP,74,"Echo (ping) reply id=0x0001, seq=412/39937,..."
2,150,16.552307,16.552307,16.622112,0.069805,-0.873639,0.864440,1.738079,192.168.0.200,216.239.38.120,ICMP,74,"Echo (ping) request id=0x0001, seq=413/40193,..."
3,151,16.622112,16.622112,17.565556,0.943444,0.864440,-0.849893,1.714333,216.239.38.120,192.168.0.200,ICMP,74,"Echo (ping) reply id=0x0001, seq=413/40193,..."
4,159,17.565556,17.565556,17.644559,0.079004,-0.849893,0.883103,1.732996,192.168.0.200,216.239.38.120,ICMP,74,"Echo (ping) request id=0x0001, seq=414/40449,..."


In [201]:
df_filtered.to_csv(TEST_OUTPUT, index=False)

In [202]:
raise StopIteration("Stop")

StopIteration: Stop

In [ ]:
test_1  = pd.read_csv('output/ping/ping_icmp_1_processed.csv')
test_2  = pd.read_csv('output/ping/ping_icmp_2_processed.csv')
test_3  = pd.read_csv('output/ping/ping_icmp_3_processed.csv')

In [ ]:
np.sum(test_1['Length'])

np.int64(42328)

In [ ]:
# Load raw data to get raw time spans
raw_test_1 = pd.read_csv('ping_test/ping_icmp_1.csv')
raw_test_2 = pd.read_csv('ping_test/ping_icmp_2.csv')
raw_test_3 = pd.read_csv('ping_test/ping_icmp_3.csv')


# Calculate raw time spans before any filtering
raw_time_span_1 = raw_test_1['Time'].max() - raw_test_1['Time'].min()
raw_time_span_2 = raw_test_2['Time'].max() - raw_test_2['Time'].min()
raw_time_span_3 = raw_test_3['Time'].max() - raw_test_3['Time'].min()

print(f"Raw Time Span Test 1: {raw_time_span_1:.6f} seconds")
print(f"Raw Time Span Test 2: {raw_time_span_2:.6f} seconds")
print(f"Raw Time Span Test 3: {raw_time_span_3:.6f} seconds")

Raw Time Span Test 1: 300.956938 seconds
Raw Time Span Test 2: 310.722727 seconds
Raw Time Span Test 3: 301.680327 seconds


In [ ]:
len(raw_test_1)

572

In [ ]:
raw_test_1.head()

,No.,Time,Source,Destination,Protocol,Length,Info
0,3,0.282538,192.168.2.144,216.239.38.120,ICMP,74,"Echo (ping) request id=0x0001, seq=28/7168, t..."
1,4,0.314824,216.239.38.120,192.168.2.144,ICMP,74,"Echo (ping) reply id=0x0001, seq=28/7168, t..."
2,12,1.293473,192.168.2.144,216.239.38.120,ICMP,74,"Echo (ping) request id=0x0001, seq=29/7424, t..."
3,13,1.597886,216.239.38.120,192.168.2.144,ICMP,74,"Echo (ping) reply id=0x0001, seq=29/7424, t..."
4,15,2.303138,192.168.2.144,216.239.38.120,ICMP,74,"Echo (ping) request id=0x0001, seq=30/7680, t..."


In [ ]:
raw_test_1['Protocol'].value_counts()

Protocol
ICMP    572
Name: count, dtype: int64

In [ ]:
test_1['Protocol'].value_counts()

Protocol
ICMP    572
Name: count, dtype: int64

In [ ]:
throughput_1 = (np.sum(test_1['Length'] * 8) / raw_time_span_1) / 1000
throughput_2 = (np.sum(test_2['Length'] * 8) / raw_time_span_2) / 1000
throughput_3 = (np.sum(test_3['Length'] * 8) / raw_time_span_3) / 1000

print(f"Throughput Test 1: {throughput_1:.6f} kbps")
print(f"Throughput Test 2: {throughput_2:.6f} kbps")
print(f"Throughput Test 3: {throughput_3:.6f} kbps")

Throughput Test 1: 1.125158 kbps
Throughput Test 2: 0.712558 kbps
Throughput Test 3: 1.006681 kbps


# Packet Loss Hitung Sendiri!

In [ ]:
# packet_loss_test_1 = pd.read_csv('browser_test/browsing_tcp_lost_segment_1.csv')
# packet_loss_test_2 = pd.read_csv('browser_test/browsing_tcp_lost_segment_2.csv')
# packet_loss_test_3 = pd.read_csv('browser_test/browsing_tcp_lost_segment_3.csv')

In [ ]:
# packet_loss_1 = len(raw_test_1) - len(packet_loss_test_1) / len(raw_test_1) * 100
# packet_loss_2 = len(raw_test_2) - len(packet_loss_test_2) / len(raw_test_2) * 100
# packet_loss_3 = len(raw_test_3) - len(packet_loss_test_3) / len(raw_test_3) * 100

In [ ]:
total_delay_test_1 = np.sum(test_1['Delay'].dropna())
total_delay_test_2 = np.sum(test_2['Delay'].dropna())
total_delay_test_3 = np.sum(test_3['Delay'].dropna())

In [ ]:
print(total_delay_test_1)

300.9569377


In [ ]:
packet_loss = 0

In [ ]:

delay_avg_1 = total_delay_test_1 / len(raw_test_1)
delay_avg_2 = total_delay_test_2 / len(raw_test_2)
delay_avg_3 = total_delay_test_3 / len(raw_test_3)

In [ ]:
total_jitter_test_1 = np.sum(test_1['Jitter'].dropna())
total_jitter_test_2 = np.sum(test_2['Jitter'].dropna())
total_jitter_test_3 = np.sum(test_3['Jitter'].dropna())

In [ ]:
print(total_jitter_test_1)

1079.7329148999997


In [ ]:
jitter_1 = (total_jitter_test_1 / len(test_1)) * 1000
jitter_2 = (total_jitter_test_2 / len(test_2)) * 1000
jitter_3 = (total_jitter_test_3 / len(test_3)) * 1000

In [ ]:
average_jitter_test_1 = total_jitter_test_1 / len(raw_test_1)
average_jitter_test_2 = total_jitter_test_2 / len(raw_test_2)
average_jitter_test_3 = total_jitter_test_3 / len(raw_test_3)

In [ ]:
output_df = pd.DataFrame({
    'Test': ['Test 1', 'Test 2', 'Test 3'],
    'Raw Time Span (s)': [raw_time_span_1, raw_time_span_2, raw_time_span_3],
    'Throughput (kbps)': [throughput_1, throughput_2, throughput_3],
    'Packet Loss (%)': [packet_loss, packet_loss, packet_loss],
    'Total Delay (s)': [total_delay_test_1, total_delay_test_2, total_delay_test_3],
    'Avg Delay (s)': [delay_avg_1, delay_avg_2, delay_avg_3],
    'Total Jitter (s)': [total_jitter_test_1, total_jitter_test_2, total_jitter_test_3],
    'Avg Jitter (s)': [average_jitter_test_1, average_jitter_test_2, average_jitter_test_3],
    'Jitter (ms)': [jitter_1, jitter_2, jitter_3]
})

output_df.to_csv('output/summary.csv', index=False)
print(output_df)

     Test  Raw Time Span (s)  Throughput (kbps)  Packet Loss (%)  \
0  Test 1         300.956938           1.125158                0   
1  Test 2         310.722727           0.712558                0   
2  Test 3         301.680327           1.006681                0   

   Total Delay (s)  Avg Delay (s)  Total Jitter (s)  Avg Jitter (s)  \
0       300.956938       0.526148       1079.732915        1.887645   
1       310.722727       0.830809       1152.989174        3.082859   
2       301.680327       0.588071       1109.193510        2.162171   

   Jitter (ms)  
0  1887.644956  
1  3082.858755  
2  2162.170584  
